In [ ]:
from huggingface_hub import login
hf_token = ""
login(token=hf_token)

In [ ]:
import torch
import json
import os
import sys
from PIL import Image
from transformers import Gemma3ForConditionalGeneration, AutoProcessor
from datasets import load_dataset
from IPython.display import FileLink

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
teacherModelName = "google/gemma-3-4b-it"

teacherModel = Gemma3ForConditionalGeneration.from_pretrained(
    teacherModelName,
    torch_dtype=torch.bfloat16,
    device_map="balanced"
)
processor = AutoProcessor.from_pretrained(teacherModelName)
print(f"GPUs detected: {torch.cuda.device_count()}")

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Describe the image in one sentence. Do not include any introductory phrases."}
        ]
    }
]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("lmms-lab/COCO-Caption2017", split="val")

In [ ]:
def generate_caption(image):
    with torch.no_grad():
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch.bfloat16)
        output_ids = teacherModel.generate(
            **inputs,
            max_new_tokens=40, 
            do_sample=False   
        )
        return processor.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
dataset_for_distillation = []
output_file = "distilled_dataset.json"

for i in range(len(dataset)):
    try:
        sample = dataset[i]
        image = sample['image']
        
        teachers_output = generate_caption(image)
        
        dataset_for_distillation.append({
            "image_id": sample['question_id'],
            "real_caption": sample['answer'][1] if len(sample['answer']) > 1 else sample['answer'][0],
            "teacher_caption": teachers_output
        })
        
        if i % 1 == 0:
            print(f"processed {i}/{len(dataset)}")
            
        if i % 100 == 0:
            with open(output_file, "w") as f:
                json.dump(dataset_for_distillation, f, indent=4)
                
    except Exception as e:
        print(f"\nError at index {i}: {e}")
        continue


In [ ]:
with open(output_file, "w") as f:
    json.dump(dataset_for_distillation, f, indent=4)

print("\nProcessing Complete! JSON file created.")